In [ ]:
import pandas as pd
import numpy as np

class SVODAnalyzer:
    """
    Complete SVOD Market Analysis Tool
    Calculates all 8 metrics + advanced insights
    """

    def __init__(self, df):
        self.df = df.copy()
        self.df['Fact_date'] = pd.to_datetime(self.df['Fact_date'])
        self.df['Year'] = self.df['Fact_date'].dt.year
        self.df['Quarter'] = self.df['Fact_date'].dt.quarter
        self.df['Year_Quarter'] = self.df['Year'].astype(str) + '-Q' + self.df['Quarter'].astype(str)

    # ====== CORE METRICS ======

    def metric_1_avg_percent_increase(self, period='annual'):
        """Metric 1: Average % increase across all actors"""
        if period == 'annual':
            start_date, end_date = '2021-12-31', '2022-12-31'

        start_data = self.df[self.df['Fact_date'] == pd.to_datetime(start_date)]
        end_data = self.df[self.df['Fact_date'] == pd.to_datetime(end_date)]

        common_actors = set(start_data['Actor_label']) & set(end_data['Actor_label'])
        pct_changes = []

        for actor in common_actors:
            start_val = start_data[start_data['Actor_label'] == actor]['Kpi_value'].values
            end_val = end_data[end_data['Actor_label'] == actor]['Kpi_value'].values

            if len(start_val) > 0 and len(end_val) > 0 and start_val[0] > 0:
                pct_change = ((end_val[0] - start_val[0]) / start_val[0]) * 100
                pct_changes.append(pct_change)

        return round(np.mean(pct_changes), 2) if pct_changes else 0

    def metric_2_actors_with_increase(self, period='annual'):
        """Metric 2: Count of actors that gained subscribers"""
        change_dict = self._get_actors_change(period)
        return len([a for a, c in change_dict.items() if c > 0])

    def metric_3_actors_with_decrease(self, period='annual'):
        """Metric 3: Count of actors that lost subscribers"""
        change_dict = self._get_actors_change(period)
        return len([a for a, c in change_dict.items() if c < 0])

    def metric_4_pct_actors_increase(self, period='annual'):
        """Metric 4: % of actors with increase"""
        m2 = self.metric_2_actors_with_increase(period)
        m3 = self.metric_3_actors_with_decrease(period)
        total = m2 + m3
        return round((m2 / total * 100), 2) if total > 0 else 0

    def metric_5_pct_actors_decrease(self, period='annual'):
        """Metric 5: % of actors with decrease"""
        m2 = self.metric_2_actors_with_increase(period)
        m3 = self.metric_3_actors_with_decrease(period)
        total = m2 + m3
        return round((m3 / total * 100), 2) if total > 0 else 0

    def metric_6_actors_by_year(self):
        """Metric 6: Count of unique actors in 2021 vs 2022"""
        actors_2021 = len(self.df[self.df['Year'] == 2021]['Actor_label'].unique())
        actors_2022 = len(self.df[self.df['Year'] == 2022]['Actor_label'].unique())
        return {
            '2021': actors_2021,
            '2022': actors_2022,
            'net_change': actors_2022 - actors_2021
        }

    def metric_7_services_stopped(self):
        """Metric 7: Services that disappeared in 2022"""
        actors_2021 = set(self.df[self.df['Year'] == 2021]['Actor_label'].unique())
        actors_2022 = set(self.df[self.df['Year'] == 2022]['Actor_label'].unique())
        stopped = actors_2021 - actors_2022
        return {
            'count': len(stopped),
            'services': list(stopped)
        }

    def metric_8_services_launched(self):
        """Metric 8: New services in 2022"""
        actors_2021 = set(self.df[self.df['Year'] == 2021]['Actor_label'].unique())
        actors_2022 = set(self.df[self.df['Year'] == 2022]['Actor_label'].unique())
        launched = actors_2022 - actors_2021
        return {
            'count': len(launched),
            'services': list(launched)
        }

    # ====== ADVANCED METRICS ======

    def weighted_market_growth(self):
        """Total market growth (not average)"""
        total_2021 = self.df[self.df['Fact_date'] == pd.to_datetime('2021-12-31')]['Kpi_value'].sum()
        total_2022 = self.df[self.df['Fact_date'] == pd.to_datetime('2022-12-31')]['Kpi_value'].sum()
        growth = ((total_2022 - total_2021) / total_2021 * 100) if total_2021 > 0 else 0
        return {
            'total_2021': total_2021,
            'total_2022': total_2022,
            'growth_pct': round(growth, 2)
        }

    def market_concentration_top3(self):
        """Market share of top 3 in each year"""
        data_2021 = self.df[self.df['Year'] == 2021].groupby('Actor_label')['Kpi_value'].sum().sort_values(ascending=False)
        data_2022 = self.df[self.df['Year'] == 2022].groupby('Actor_label')['Kpi_value'].sum().sort_values(ascending=False)

        top3_share_2021 = (data_2021.head(3).sum() / data_2021.sum() * 100) if data_2021.sum() > 0 else 0
        top3_share_2022 = (data_2022.head(3).sum() / data_2022.sum() * 100) if data_2022.sum() > 0 else 0

        return {
            'top3_2021_share': round(top3_share_2021, 2),
            'top3_2022_share': round(top3_share_2022, 2),
            'concentration_change': round(top3_share_2022 - top3_share_2021, 2),
            'story': 'Consolidating' if (top3_share_2022 - top3_share_2021) > 0 else 'Diversifying'
        }

    def top_gainers_decliners(self, top_n=5):
        """Top gainers and decliners"""
        change_dict = self._get_actors_change('annual')
        sorted_actors = sorted(change_dict.items(), key=lambda x: x[1], reverse=True)

        return {
            'gainers': sorted_actors[:top_n],
            'decliners': sorted_actors[-top_n:][::-1]
        }

    def qoq_growth_analysis(self):
        """Quarter-over-quarter growth rates"""
        market_q = self.df.groupby('Year_Quarter')['Kpi_value'].sum().reset_index()
        market_q['QoQ_Growth_Pct'] = market_q['Kpi_value'].pct_change() * 100
        return market_q

    # ====== HELPER ======

    def _get_actors_change(self, period='annual'):
        """Helper: Get subscriber change for each actor"""
        if period == 'annual':
            start_date, end_date = '2021-12-31', '2022-12-31'

        start_data = self.df[self.df['Fact_date'] == pd.to_datetime(start_date)]
        end_data = self.df[self.df['Fact_date'] == pd.to_datetime(end_date)]

        change_dict = {}
        for actor in set(start_data['Actor_label']) & set(end_data['Actor_label']):
            start_val = start_data[start_data['Actor_label'] == actor]['Kpi_value'].values[0]
            end_val = end_data[end_data['Actor_label'] == actor]['Kpi_value'].values[0]
            change_dict[actor] = end_val - start_val

        return change_dict

    # ====== REPORTING ======

    def print_full_report(self):
        """Print complete analysis"""
        print("="*80)
        print("SVOD MARKET EVOLUTION: USA 2021-2022")
        print("="*80)
        print()

        print("CORE METRICS:")
        print(f"  1. Avg % Increase: {self.metric_1_avg_percent_increase()}%")
        print(f"  2. Actors with Increase: {self.metric_2_actors_with_increase()}")
        print(f"  3. Actors with Decrease: {self.metric_3_actors_with_decrease()}")
        print(f"  4. % Actors Increasing: {self.metric_4_pct_actors_increase()}%")
        print(f"  5. % Actors Decreasing: {self.metric_5_pct_actors_decrease()}%")

        m6 = self.metric_6_actors_by_year()
        print(f"  6. Actor Count: 2021={m6['2021']}, 2022={m6['2022']}, Net={m6['net_change']:+d}")

        m7 = self.metric_7_services_stopped()
        print(f"  7. Services Stopped: {m7['count']}")
        if m7['services']:
            print(f"     → {', '.join(m7['services'])}")

        m8 = self.metric_8_services_launched()
        print(f"  8. Services Launched: {m8['count']}")
        if m8['services']:
            print(f"     → {', '.join(m8['services'])}")

        print()
        print("ADVANCED INSIGHTS:")

        growth = self.weighted_market_growth()
        print(f"\n  Weighted Growth:")
        print(f"    2021 Total: {growth['total_2021']:,.0f}")
        print(f"    2022 Total: {growth['total_2022']:,.0f}")
        print(f"    Growth: {growth['growth_pct']}%")

        conc = self.market_concentration_top3()
        print(f"\n  Market Concentration (Top 3):")
        print(f"    2021 Share: {conc['top3_2021_share']}%")
        print(f"    2022 Share: {conc['top3_2022_share']}%")
        print(f"    Change: {conc['concentration_change']:+.2f}pp → {conc['story']}")

        top = self.top_gainers_decliners(5)
        print(f"\n  Top 5 Gainers:")
        for actor, change in top['gainers']:
            print(f"    {actor}: +{change:,.0f}")

        print(f"\n  Top 5 Decliners:")
        for actor, change in top['decliners']:
            print(f"    {actor}: {change:,.0f}")

        print()
        print("="*80)


# ============================================================================
# USAGE
# ============================================================================

if __name__ == "__main__":
    # Load your data
    df = pd.read_csv('Data.csv')

    # Run analysis
    analyzer = SVODAnalyzer(df)
    analyzer.print_full_report()

    # Export for dashboard
    timeline = analyzer.df.sort_values('Fact_date')
    timeline.to_csv('svod_timeline_continuous.csv', index=False)

    qoq = analyzer.qoq_growth_analysis()
    qoq.to_csv('qoq_growth_rates.csv', index=False)

    print("\nExported: svod_timeline_continuous.csv, qoq_growth_rates.csv")

SVOD MARKET EVOLUTION: USA 2021-2022

CORE METRICS:
  1. Avg % Increase: 36.56%
  2. Actors with Increase: 106
  3. Actors with Decrease: 8
  4. % Actors Increasing: 92.98%
  5. % Actors Decreasing: 7.02%
  6. Actor Count: 2021=127, 2022=122, Net=-5
  7. Services Stopped: 9
     → WWE Network, WatchTV, Rabbit TV Plus, AT&T TV Now, NHL.TV, TVision, BossTV, Spuul, Bleacher Report Live
  8. Services Launched: 4
     → Vix Premium, NFL+, Bally Sports+, iQiYi

ADVANCED INSIGHTS:

  Weighted Growth:
    2021 Total: 442,528,032
    2022 Total: 510,268,406
    Growth: 15.31%

  Market Concentration (Top 3):
    2021 Share: 49.68%
    2022 Share: 43.57%
    Change: -6.10pp → Diversifying

  Top 5 Gainers:
    Paramount+: +14,981,000
    Peacock Premium: +11,915,408
    Amazon Prime Video: +5,245,681
    Apple TV+: +5,080,667
    YouTube Premium: +4,950,506

  Top 5 Decliners:
    Netflix: -795,171
    Showtime Streaming: -433,674
    Sling TV: -152,000
    C Spire: -13,174
    PBS Passport: -6,

In [ ]:
import pandas as pd

excel_file_path = 'video_on_demand.xlsx'

# Create an ExcelFile object
xcel = pd.ExcelFile(excel_file_path)

# Get all sheet names
sheet_names = xcel.sheet_names

print(f"Found {len(sheet_names)} sheets in '{excel_file_path}': {sheet_names}")

# Iterate through each sheet and save it as a separate CSV
for sheet_name in sheet_names:
    df = xcel.parse(sheet_name)
    output_filename = f'{sheet_name}.csv'
    df.to_csv(output_filename, index=False)
    print(f"Saved sheet '{sheet_name}' to '{output_filename}'")


Found 2 sheets in 'Dataxis_Test BI Analyst 2026.xlsx': ['Exercise', 'Data']
Saved sheet 'Exercise' to 'Exercise.csv'
Saved sheet 'Data' to 'Data.csv'
